## **Baseline Model Evaluation**

This notebook evaluates the winning solution from the [AICrowd Airborne Object Tracking Challenge](https://www.aicrowd.com/challenges/airborne-object-tracking-challenge) (2021) as the project baseline. The solution, developed by Dmytro Poplavskiy, uses a segmentation-based detection and tracking pipeline (SegTracker) with an ensemble of four deep learning models: HRNet32, HRNet48, GerNet-M with BiFPN, and DLA60.

Inference is performed across the entire custom hard dataset subset [outlined below](#dataset), with results presented both as an overall set of benchmark statistics and as performance metrics for each data subset.

**Evaluation Metrics:**
- **AFDR:** Airborne Flight Detection Rate. The primary detection metric.
- **FPPI:** False Positives Per Image. Must be ≤ 0.0005 to be ranked.
- **HFAR:** Horizontal False Alarm Rate. Must be ≤ 0.5 to be ranked.
- **EDR:** Encounter Detection Rate. Tracking-based metric across all horizon scenarios.

> ⚠️ ***Note:*** *Inference requires a CUDA-capable GPU. The `map_location` fix in `seg_tracker.py` allows CPU loading of weights for inspection, but full inference at acceptable speed requires GPU. Estimated runtime on GPU: ~2–3 FPS per flight sequence.*

In [1]:
# Import Required Libraries.
import os
import sys

import json
import subprocess

import pandas as pd
from pathlib import Path

from glob import glob
from logging_module import LogLevel, log

# Set Working Directory to Project Root.
os.chdir("..") 

# Add Project Root to Path.
sys.path.insert(0, os.getcwd())

# Add Core Metric Folders to System Path.
sys.path.insert(0, os.path.join(os.getcwd(), "core/metrics"))


---

<br/>

## **Dataset**

The custom dataset used in this notebook is a curated hard subset of the full Airborne Object Tracking (AOT) dataset, specifically designed to stress-test model performance under the most demanding real-world conditions. Flights were selected to isolate encounters that represent the greatest challenge for vision-based detection and tracking systems.

**Isolation Criteria - encounters were selected for:**
- **Small Targets:** Airborne objects occupying only a few pixels per frame.
- **Multiple Targets:** Scenes containing more than one airborne object simultaneously.
- **Poor Visibility:** Flights affected by sun glare, adverse lighting, or challenging camera angles.

Together these conditions reflect the edge cases where sense-and-avoid systems are most likely to fail, and where reliable early detection is most critical for safe autonomous flight.

### **Airborne Object Tracking (AOT) - Hard Subset Summary**

| Metric                   | Value   | Description                                                             |
|--------------------------|--------:|-------------------------------------------------------------------------|
| **Total Flights**        | **1618**| **Total number of flight sequences in the dataset**                     |
| Small Object Flights     | 662     | Flights containing at least one object with bounding box area ≤ 36.0px².|
| Below Horizon Flights    | 1053    | Flights containing at least one object at or below the horizon.         |
| Multi-Object Flights     | 627     | Flights containing more than one airborne object in at least one frame. |
| No Object Frames         | 354     | Flights containing at least one frame with no airborne object present.  |
| *Avg. Sequence Length*   | *17.77s*| *Average duration of each flight sequence.*                             |

In [ ]:
# Import Data Downloader Libraries.
from src.data.download_data import main as downloadData

# Set Path to Desired Evaluation Dataset.
DATASET_PATH = "configs/datasets/hard_dataset_flights.txt"

# Download the Selected Flight Sequences.
downloadData(DATASET_PATH)

In [2]:
# Set Dataset Ground Truth Folders.
DATASET_PARTS = [
    "data/part1/ImageSets",
    "data/part2/ImageSets",
    "data/part3/ImageSets",
]

# Set Corresponding Inference Image Folders.
IMAGE_PARTS = [
    "data/part1/Images",
    "data/part2/Images",
    "data/part3/Images",
]

# Set Data Folder Names Used During Inference.
DATASET_ENVS = [
    "run_part1",
    "run_part2",
    "run_part3",
]

# Set Output Results Folders (Must Match DATASET_ENV Names Above).
RESULTS_DIRS = [
    "data/results/baseline/run_part1",
    "data/results/baseline/run_part2",
    "data/results/baseline/run_part3",
]

# Aggregated Summaries Output Folder.
SUMMARIES_DIR = "data/results/baseline/summaries"

# Path to the Metrics Runner Script.
METRICS_SCRIPT = "core/metrics/run_airborne_metrics.py"

# Minimum Score Threshold.
MIN_SCORE = 0.985

# Minimum Track Length Passed to Metrics Script.
# NOTE: Track length filtering is already applied during inference. Pass 0 here 
# to avoid double-filtering which causes NaN assertion errors in the metrics.
MIN_TRACK_LEN = 0

# Print Summary Statement.
log(LogLevel.INFO, "baseline:config", "Configuration Loaded.")
print(f"  Dataset Parts : {len(DATASET_PARTS)}")
print(f"  Results Dirs  : {RESULTS_DIRS}")
print(f"  Summaries Dir : {SUMMARIES_DIR}")
print(f"  Min Score     : {MIN_SCORE}")
print(f"  Min Track Len : {MIN_TRACK_LEN}")

<INFO ><baseline:config> Configuration Loaded.
  Dataset Parts : 3
  Results Dirs  : ['data/results/baseline/run_part1', 'data/results/baseline/run_part2', 'data/results/baseline/run_part3']
  Summaries Dir : data/results/baseline/summaries
  Min Score     : 0.985
  Min Track Len : 0



---

<br/>

## **Evaluate Baseline Model**

### **Run Baseline Inference**

Runs `seg_test.py` across each dataset part sequentially, saving results to the configured output directories.

> ⚠️ **This cell requires a CUDA GPU and will take several hours to complete.** Skip this cell if inference has already been run and results are present in `RESULTS_DIRS`.

In [3]:
# Import Data Downloader Libraries.
from core.metrics.run_airborne_metrics import get_next_run_dir

# Run Inference On Each Dataset Part.
for i, (image_dir, results_dir) in enumerate(zip(IMAGE_PARTS, RESULTS_DIRS)):
    # Set Part Name.
    part_name = f"Part {i+1}"

    # Auto-Increment Run Folder so Previous Results are Never Overwritten.
    run_name = get_next_run_dir(results_dir)

    # Print Current Run Info.
    log(LogLevel.INFO, "baseline:inference", "Running Inference on %s...", part_name)
    print(f"  Images  : {image_dir}")
    print(f"  Results : {results_dir}")
    print(f"  Run Folder : {run_name}")
    print(f"{'-'*58}")

    # Ensure Results Directory Exists. 
    os.makedirs(results_dir, exist_ok=True)

    # Set the Variables to Correct Paths.
    env = os.environ.copy()
    env["TEST_DATASET_PATH"] = image_dir
    env["INFERENCE_OUTPUT_PATH"] = results_dir
    env["DATASET_ENV"] = run_name

    # Run the Basline Model Inference.
    result = subprocess.run(
        [sys.executable, "-m", "src.baseline.seg_test"],
        env=env,
        capture_output=False
    )

    # Print Message on Completion. 
    if result.returncode == 0:
        log(LogLevel.INFO, "baseline:inference", "%s Inference Complete.\n", part_name)
    else:
        log(LogLevel.ERROR, "baseline:inference", "%s Inference Failed with Return Code %s.\n", part_name, result.returncode)

<INFO ><baseline:inference> Running Inference on Part 1...
  Images  : data/part1/Images
  Results : data/results/baseline/run_part1
  Run Folder : run1
----------------------------------------------------------


/home/maxbusato/Projects/Vision-Based-Sense-and-Avoid/.venv/lib/python3.8/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


hrnet_w32 upscale: nearest comb outputs: 512
Feature channels: [32, 64, 128, 256]
Using custom backbone gernet_m
{'name': 'tf_efficientdet_d2', 'backbone_name': 'gernet_m', 'backbone_args': {'in_chans': 2}, 'backbone_indices': None, 'image_size': [512, 512], 'num_classes': 2, 'min_level': 3, 'max_level': 7, 'num_levels': 5, 'num_scales': 3, 'aspect_ratios': [[1.0, 1.0], [1.4, 0.7], [0.7, 1.4]], 'anchor_scale': 4.0, 'pad_type': 'same', 'act_type': 'swish', 'norm_layer': None, 'norm_kwargs': {'eps': 0.001, 'momentum': 0.01}, 'box_class_repeats': 3, 'fpn_cell_repeats': 5, 'fpn_channels': 112, 'separable_conv': True, 'apply_resample_bn': True, 'conv_after_downsample': False, 'conv_bn_relu_pattern': False, 'use_native_resize_op': False, 'downsample_type': 'max', 'upsample_type': 'nearest', 'redundant_bias': True, 'head_bn_level_first': False, 'head_act_type': None, 'fpn_name': None, 'fpn_config': None, 'fpn_drop_path_rate': 0.0, 'alpha': 0.25, 'gamma': 1.5, 'label_smoothing': 0.0, 'legacy_f

100%|██████████| 89/89 [00:30<00:00,  2.96it/s]


<INFO ><baseline:inference> Part 1 Inference Complete.

<INFO ><baseline:inference> Running Inference on Part 2...
  Images  : data/part2/Images
  Results : data/results/baseline/run_part2
  Run Folder : run1
----------------------------------------------------------


/home/maxbusato/Projects/Vision-Based-Sense-and-Avoid/.venv/lib/python3.8/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


... (3 outputs left out)

/home/maxbusato/Projects/Vision-Based-Sense-and-Avoid/.venv/lib/python3.8/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


hrnet_w32 upscale: nearest comb outputs: 512
Feature channels: [32, 64, 128, 256]
Using custom backbone gernet_m
{'name': 'tf_efficientdet_d2', 'backbone_name': 'gernet_m', 'backbone_args': {'in_chans': 2}, 'backbone_indices': None, 'image_size': [512, 512], 'num_classes': 2, 'min_level': 3, 'max_level': 7, 'num_levels': 5, 'num_scales': 3, 'aspect_ratios': [[1.0, 1.0], [1.4, 0.7], [0.7, 1.4]], 'anchor_scale': 4.0, 'pad_type': 'same', 'act_type': 'swish', 'norm_layer': None, 'norm_kwargs': {'eps': 0.001, 'momentum': 0.01}, 'box_class_repeats': 3, 'fpn_cell_repeats': 5, 'fpn_channels': 112, 'separable_conv': True, 'apply_resample_bn': True, 'conv_after_downsample': False, 'conv_bn_relu_pattern': False, 'use_native_resize_op': False, 'downsample_type': 'max', 'upsample_type': 'nearest', 'redundant_bias': True, 'head_bn_level_first': False, 'head_act_type': None, 'fpn_name': None, 'fpn_config': None, 'fpn_drop_path_rate': 0.0, 'alpha': 0.25, 'gamma': 1.5, 'label_smoothing': 0.0, 'legacy_f

100%|██████████| 209/209 [01:01<00:00,  3.42it/s]


<INFO ><baseline:inference> Part 3 Inference Complete.



### **Run Metrics Evaluation**

Runs the official AICrowd evaluation metrics against the ground truth for each dataset part, saving per-part summaries to `SUMMARIES_DIR`.

In [5]:
# Ensure the Summaries Directory Exists. 
os.makedirs(SUMMARIES_DIR, exist_ok=True)

# Run Metrics for Each Part.
for i, (dataset_dir, results_dir) in enumerate(zip(DATASET_PARTS, RESULTS_DIRS)):
    # Set Part Name.
    part_name = f"Part {i+1}"

    # Set Results Folder Name.
    results_file_dir = os.path.join(results_dir, run_name)
    
    # Print Current Run Info.
    log(LogLevel.INFO, "baseline:evaluation", "Evaluating %s...", part_name)
    print(f"  Ground Truth : {dataset_dir}")
    print(f"  Results      : {results_file_dir}")

    # Check Results Exist Before Evaluating.
    result_files = (
        glob(os.path.join(results_file_dir, "*.json")) +
        glob(os.path.join(results_file_dir, "*.csv"))
    )
    if not result_files:
        log(LogLevel.WARNING, "baseline:evaluation", "No Result Files Found in %s - Skipping.", results_file_dir)
        log(LogLevel.WARNING, "baseline:evaluation", "Run Inference Module First.")
        continue

    # Save Results for Each Part Separately.
    part_summaries_dir = os.path.join(SUMMARIES_DIR, part_name)
    os.makedirs(part_summaries_dir, exist_ok=True)

    # Run the Metrics Calculating Script. 
    result = subprocess.run(
        [
            sys.executable, METRICS_SCRIPT,
            "--dataset-folder", dataset_dir,
            "--results-folder", results_file_dir,
            "--summaries-folder", part_summaries_dir,
            "--min-score", str(MIN_SCORE),
            "--min-track-len", str(MIN_TRACK_LEN),
        ],
        capture_output=True,
        text=True
    )

    # Print Message on Completion. 
    if result.returncode == 0:
        log(LogLevel.INFO, "baseline:evaluation", "%s Metrics Complete.\n", part_name)
    else:
        log(LogLevel.ERROR, "baseline:inference", "%s Metrics Failed with Error:\nSTDOUT:\n%s\nSTDERR:\n%s\n", 
            part_name, result.stdout, result.stderr)

<INFO ><baseline:evaluation> Evaluating Part 1...
  Ground Truth : data/part1/ImageSets
  Results      : data/results/baseline/run_part1/run1
<ERROR><baseline:inference> Part 1 Metrics Failed with Error:
STDOUT:

STDERR:
2026-03-20 14:34:29,404:INFO:run_airborne_metrics.py:166 Encounter ground truth: data/part1/ImageSets/groundtruth_with_encounters_maxRange700_maxGap3_minEncLen30.csv
2026-03-20 14:34:29,405:INFO:match_groundtruth_results.py:516 Reading input ground truth and results
2026-03-20 14:34:29,405:INFO:pandas_utils.py:87 Reading ground truth
2026-03-20 14:34:29,405:INFO:pandas_utils.py:61 Reading provided data/part1/ImageSets/groundtruth.csv
2026-03-20 14:34:32,986:INFO:match_groundtruth_results.py:522 Number of evaluated images is 1570557
2026-03-20 14:34:32,986:INFO:pandas_utils.py:96 Reading detection results
2026-03-20 14:34:32,986:INFO:pandas_utils.py:61 Reading provided data/results/baseline/run_part1/run1/result.csv
2026-03-20 14:34:32,988:INFO:match_groundtruth_results

KeyboardInterrupt: 